In [1]:
!pip install -q langchain-text-splitters langchain-huggingface langchain-chroma

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import json


In [3]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
DATASET_PATH = "/content/songs_dataset_50_final.json"

def load_dataset(path):
    with open(path, "r", encoding="utf-8") as file:
        dataset = json.load(file)
    return dataset

dataset = load_dataset(DATASET_PATH)

print(f"Loaded {len(dataset)} songs.")
dataset[0]

Loaded 50 songs.


{'id': 1,
 'song_name': 'Shape of You',
 'artist': 'Ed Sheeran',
 'genre': 'Pop',
 'mood': 'Romantic',
 'tempo': 'Medium',
 'language': 'English',
 'release_year': 2017,
 'description': 'This Pop song by Ed Sheeran delivers a romantic listening experience with a medium tempo. Released in 2017, it blends memorable melodies, polished production, and expressive vocals to create a distinctive atmosphere. The arrangement balances rhythm, harmony, and emotion, making it enjoyable from the first listen while revealing more details over repeated plays. Listeners often associate the track with moments such as road trips, evening drives, studying, relaxing, celebrations, workouts, or spending time with friends depending on its mood and energy. Its musical style fits naturally alongside other songs in the Pop genre and appeals to fans looking for similar sounds, instrumentation, and emotional themes. The lyrics explore ideas connected to romantic emotions, personal experiences, relationships, and

In [5]:
def create_documents(dataset):
    documents = []

    for song in dataset:
        text = f"""
Song Name: {song['song_name']}
Artist: {song['artist']}
Genre: {song['genre']}
Mood: {song['mood']}
Tempo: {song['tempo']}
Language: {song['language']}
Release Year: {song['release_year']}

Description:
{song['description']}
"""

        document = Document(
            page_content=text,
            metadata={
                "id": song["id"],
                "song_name": song["song_name"],
                "artist": song["artist"],
                "genre": song["genre"],
                "mood": song["mood"],
                "tempo": song["tempo"],
                "language": song["language"],
                "release_year": song["release_year"]
            }
        )

        documents.append(document)

    return documents

In [6]:
documents = create_documents(dataset)

print(f"Total Documents: {len(documents)}")
print(documents[0])

Total Documents: 50
page_content='
Song Name: Shape of You
Artist: Ed Sheeran
Genre: Pop
Mood: Romantic
Tempo: Medium
Language: English
Release Year: 2017

Description:
This Pop song by Ed Sheeran delivers a romantic listening experience with a medium tempo. Released in 2017, it blends memorable melodies, polished production, and expressive vocals to create a distinctive atmosphere. The arrangement balances rhythm, harmony, and emotion, making it enjoyable from the first listen while revealing more details over repeated plays. Listeners often associate the track with moments such as road trips, evening drives, studying, relaxing, celebrations, workouts, or spending time with friends depending on its mood and energy. Its musical style fits naturally alongside other songs in the Pop genre and appeals to fans looking for similar sounds, instrumentation, and emotional themes. The lyrics explore ideas connected to romantic emotions, personal experiences, relationships, and self-expression w

In [7]:
def create_vector_db(documents):
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory="./songs_db"
    )

    return vectorstore
vectorstore = create_vector_db(documents)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [8]:
#test 1
results = vectorstore.similarity_search(
    "A romantic English pop song with emotional lyrics",
    k=5
)

for i, doc in enumerate(results, 1):
    print(f"\n------ Result {i} ------")
    print(doc.metadata["song_name"])
    print(doc.metadata["artist"])
    print(doc.metadata["genre"])
    print(doc.metadata["mood"])


------ Result 1 ------
Lovely
Billie Eilish & Khalid
Indie Pop
Melancholic

------ Result 2 ------
Kesariya
Arijit Singh
Bollywood
Romantic

------ Result 3 ------
Perfect
Ed Sheeran
Pop
Romantic

------ Result 4 ------
Bekhayali
Sachet Tandon
Bollywood
Heartbroken

------ Result 5 ------
Galliyan
Ankit Tiwari
Bollywood
Romantic


#Test 2
query = documents[0].page_content

results = vectorstore.similarity_search_with_score(
    query,
    k=6
)

for doc, score in results:
    print(doc.metadata["song_name"], score)

since this resulted in redundancy , so i would skip chunking entierly

In [10]:
def search_songs(query):
    results = vectorstore.similarity_search_with_score(
        query=query,
        k=5
    )

    recommendations = []

    for doc, score in results:
        recommendations.append({
            "Song Name": doc.metadata["song_name"],
            "Artist": doc.metadata["artist"],
            "Genre": doc.metadata["genre"],
            "Mood": doc.metadata["mood"],
            "Similarity Score": round(score, 4)
        })

    return recommendations


query = input("Enter your music preference: ")

results = search_songs(query)

print("\nTop 5 Recommended Songs:\n")

for i, song in enumerate(results, 1):
    print(f"Recommendation {i}")
    print(f"Song Name       : {song['Song Name']}")
    print(f"Artist          : {song['Artist']}")
    print(f"Genre           : {song['Genre']}")
    print(f"Mood            : {song['Mood']}")
    print(f"Similarity Score: {song['Similarity Score']}")
    print("-" * 40)

Enter your music preference: I'm searching for beautiful Hindi love songs with emotional vocals, soothing melodies, and heartfelt lyrics. The songs should express deep love, commitment, togetherness, and emotional connection. I enjoy slow to medium tempo Bollywood music that is perfect for long drives, weddings, spending time with a loved one, or simply relaxing. Rich orchestration, meaningful lyrics, and romantic feelings are what I'm looking for.

Top 5 Recommended Songs:

Recommendation 1
Song Name       : Kesariya
Artist          : Arijit Singh
Genre           : Bollywood
Mood            : Romantic
Similarity Score: 0.7142
----------------------------------------
Recommendation 2
Song Name       : Lover
Artist          : Diljit Dosanjh
Genre           : Punjabi Pop
Mood            : Romantic
Similarity Score: 0.7201
----------------------------------------
Recommendation 3
Song Name       : Apna Bana Le
Artist          : Arijit Singh
Genre           : Bollywood
Mood            : Ro